# v3h False Negative Audit

Inference-only analysis for the current v3h YOLO baseline.

No training is performed.

Inputs:

- Dataset: `/kaggle/input/datasets/matanerdy/detection-dataset/dataset_yolo_bbox_v3h_li_manual_curated_val/dataset_yolo_bbox_v3h_li_manual_curated_val`
- Weights: `/kaggle/input/datasets/matanerdy/detection-dataset/best.pt`

Outputs:

- `gt_object_audit.csv`
- `false_negative_audit.csv`
- `false_negative_types.csv`
- `found_vs_missed_size_stats.csv`
- `reason_candidates.csv`
- `manual_review_priority_images.csv`
- `outputs/fn_crops/`
- `outputs/fn_contact_sheets/fn_page_*.jpg`
- `outputs/suspicious_fn/`
- `summary.md`


## 1. Configuration


In [ ]:
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
WORK_DATASETS_DIR = WORK_ROOT / "datasets"
OUTPUT_ROOT = WORK_ROOT / "v3h_false_negative_audit"
SCRIPT_DIR = OUTPUT_ROOT / "scripts"

DATASET_FOLDER = "dataset_yolo_bbox_v3h_li_manual_curated_val"
PREBUILT_DATASET_DIR = Path(
    "/kaggle/input/datasets/matanerdy/detection-dataset/dataset_yolo_bbox_v3h_li_manual_curated_val/dataset_yolo_bbox_v3h_li_manual_curated_val"
)
WEIGHTS_INPUT_PATH = Path("/kaggle/input/datasets/matanerdy/detection-dataset/best.pt")

DATASET_WORK_DIR = WORK_DATASETS_DIR / DATASET_FOLDER
METADATA_PATH = DATASET_WORK_DIR / "metadata.csv"

IMGSZ = 640
CONF = 0.25
ANALYSIS_CONF = 0.001
NMS_IOU = 0.50
MATCH_IOU = 0.50
NEAR_MISS_IOU = 0.30
DEVICE = 0

WORK_DATASETS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Install Dependencies


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pandas", "pillow", "matplotlib"],
    check=True,
)


## 3. Copy Dataset and Weights


In [ ]:
import shutil
import zipfile

import pandas as pd
import yaml

def find_dataset_dir(folder_name: str) -> Path | None:
    if PREBUILT_DATASET_DIR.exists():
        return PREBUILT_DATASET_DIR
    for metadata in KAGGLE_INPUT_ROOT.rglob("metadata.csv"):
        parent = metadata.parent
        if parent.name == folder_name and (parent / "images").exists() and (parent / "labels").exists():
            return parent
    return None

def find_dataset_zip(folder_name: str) -> Path | None:
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob(f"{folder_name}.zip"))
    return candidates[0] if candidates else None

source_dataset = find_dataset_dir(DATASET_FOLDER)
if DATASET_WORK_DIR.exists():
    shutil.rmtree(DATASET_WORK_DIR)

if source_dataset is not None:
    print("Copying dataset:", source_dataset)
    shutil.copytree(source_dataset, DATASET_WORK_DIR)
else:
    zip_path = find_dataset_zip(DATASET_FOLDER)
    if zip_path is None:
        raise FileNotFoundError(
            f"Attach Kaggle input containing {DATASET_FOLDER}/ or {DATASET_FOLDER}.zip"
        )
    unzip_root = WORK_DATASETS_DIR / "_unzipped_v3h"
    if unzip_root.exists():
        shutil.rmtree(unzip_root)
    unzip_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(unzip_root)
    candidates = [p.parent for p in unzip_root.rglob("metadata.csv") if p.parent.name == DATASET_FOLDER]
    extracted = candidates[0] if candidates else next(unzip_root.rglob("metadata.csv")).parent
    shutil.copytree(extracted, DATASET_WORK_DIR)

dataset_yaml = DATASET_WORK_DIR / "dataset.yaml"
dataset_yaml.write_text(
    yaml.safe_dump(
        {
            "path": str(DATASET_WORK_DIR.resolve()),
            "train": "images/train",
            "val": "images/val",
            "names": {0: "kurgan"},
        },
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)

if not WEIGHTS_INPUT_PATH.exists():
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob("best.pt"))
    if not candidates:
        raise FileNotFoundError("Attach best.pt as a Kaggle input.")
    weights_path = candidates[0]
else:
    weights_path = WEIGHTS_INPUT_PATH

metadata = pd.read_csv(METADATA_PATH)
images = metadata.drop_duplicates("image")
boxes = metadata[metadata["class_name"].notna()]
print("Dataset:", DATASET_WORK_DIR)
print("Weights:", weights_path)
print("Images:", len(images))
print("Val images:", len(images[images["split"].eq("val")]))
print("Val bbox:", len(boxes[boxes["split"].eq("val")]))
print(boxes[boxes["split"].eq("val")].groupby(["region", "source_class_name"]).size())


## 4. Write Audit Script


In [ ]:
AUDIT_SCRIPT = 'from __future__ import annotations\n\nimport argparse\nimport math\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFont\nfrom ultralytics import YOLO\n\nos.environ.setdefault("MPLBACKEND", "Agg")\nos.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")\n\ntry:\n    import matplotlib\n\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\nexcept Exception:  # pragma: no cover\n    plt = None\n\n\nMATCH_IOU = 0.50\nNEAR_MISS_IOU = 0.30\nANALYSIS_CONF = 0.001\nEDGE_MARGIN_PX = 2.0\nPAGE_SIZE = 25\nCLASS_COLORS = {\n    "kurgany_tselye": "#00ff66",\n    "kurgany_povrezhdennye": "#ffcc00",\n}\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Detailed false negative audit for v3h YOLO baseline.")\n    parser.add_argument("--metadata", type=Path, required=True)\n    parser.add_argument("--weights", type=Path, required=True)\n    parser.add_argument("--out-dir", type=Path, required=True)\n    parser.add_argument("--imgsz", type=int, default=640)\n    parser.add_argument("--conf", type=float, default=0.25)\n    parser.add_argument("--analysis-conf", type=float, default=ANALYSIS_CONF)\n    parser.add_argument("--nms-iou", type=float, default=0.50)\n    parser.add_argument("--match-iou", type=float, default=MATCH_IOU)\n    parser.add_argument("--near-miss-iou", type=float, default=NEAR_MISS_IOU)\n    parser.add_argument("--device", default=None)\n    return parser.parse_args()\n\n\ndef resolve_image_path(row: pd.Series, dataset_dir: Path) -> Path:\n    split = str(row["split"])\n    image_name = str(row.get("image_name") or Path(str(row["image"])).name)\n    candidates = [\n        dataset_dir / "images" / split / image_name,\n        Path(str(row["image"])),\n        dataset_dir / "images" / split / Path(str(row["image"])).name,\n    ]\n    for candidate in candidates:\n        if candidate.exists():\n            return candidate.resolve()\n    return candidates[0].resolve()\n\n\ndef load_val_metadata(metadata_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, list[Path]]:\n    metadata_path = metadata_path.resolve()\n    dataset_dir = metadata_path.parent\n    meta = pd.read_csv(metadata_path)\n    val = meta[meta["split"].astype(str).str.lower().eq("val")].copy()\n    if val.empty:\n        raise ValueError("No validation rows in metadata.csv")\n    if "source_class_name" not in val.columns:\n        val["source_class_name"] = val["class_name"]\n    if "source_id" not in val.columns:\n        source_cols = [col for col in ["region", "modality", "raster_file"] if col in val.columns]\n        val["source_id"] = val[source_cols].astype(str).agg("|".join, axis=1)\n\n    val["image_path"] = val.apply(lambda row: resolve_image_path(row, dataset_dir), axis=1)\n    missing = sorted({str(path) for path in val["image_path"] if not Path(path).exists()})\n    if missing:\n        raise FileNotFoundError("Missing validation images:\\n" + "\\n".join(missing[:10]))\n    val["image_key"] = val["image_path"].map(lambda p: str(Path(p).resolve()))\n\n    images = val.drop_duplicates("image_key").copy()\n    gt = val[val["class_name"].notna()].copy().reset_index(drop=True)\n    gt["gt_id"] = np.arange(len(gt))\n    gt["objects_in_tile"] = pd.to_numeric(gt.get("n_objects", np.nan), errors="coerce")\n    gt["bbox_width_px"] = pd.to_numeric(gt.get("bbox_width_px", np.nan), errors="coerce")\n    gt["bbox_height_px"] = pd.to_numeric(gt.get("bbox_height_px", np.nan), errors="coerce")\n    if gt["bbox_width_px"].isna().all():\n        gt["bbox_width_px"] = pd.to_numeric(gt["bbox_x2_px"], errors="coerce") - pd.to_numeric(gt["bbox_x1_px"], errors="coerce")\n    if gt["bbox_height_px"].isna().all():\n        gt["bbox_height_px"] = pd.to_numeric(gt["bbox_y2_px"], errors="coerce") - pd.to_numeric(gt["bbox_y1_px"], errors="coerce")\n    gt["bbox_area_px"] = pd.to_numeric(gt["bbox_area_px"], errors="coerce")\n\n    boxes = []\n    edge_flags = []\n    for _, row in gt.iterrows():\n        image = Image.open(row["image_key"])\n        width, height = image.size\n        if {"yolo_xc", "yolo_yc", "yolo_w", "yolo_h"}.issubset(gt.columns) and pd.notna(row.get("yolo_xc")):\n            xc = float(row["yolo_xc"]) * width\n            yc = float(row["yolo_yc"]) * height\n            bw = float(row["yolo_w"]) * width\n            bh = float(row["yolo_h"]) * height\n            x1, y1, x2, y2 = (xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2)\n        else:\n            x1, y1, x2, y2 = (float(row["bbox_x1_px"]), float(row["bbox_y1_px"]), float(row["bbox_x2_px"]), float(row["bbox_y2_px"]))\n        boxes.append((x1, y1, x2, y2))\n        metadata_edge = str(row.get("bbox_touches_tile_edge", "")).lower() in {"true", "1", "yes"}\n        geometric_edge = x1 <= EDGE_MARGIN_PX or y1 <= EDGE_MARGIN_PX or x2 >= width - EDGE_MARGIN_PX or y2 >= height - EDGE_MARGIN_PX\n        edge_flags.append(bool(metadata_edge or geometric_edge))\n    gt["bbox_xyxy"] = boxes\n    gt["touches_tile_edge"] = edge_flags\n    return images, gt, [Path(p).resolve() for p in images["image_path"]]\n\n\ndef box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    if len(a) == 0 or len(b) == 0:\n        return np.zeros((len(a), len(b)), dtype=float)\n    x1 = np.maximum(a[:, None, 0], b[None, :, 0])\n    y1 = np.maximum(a[:, None, 1], b[None, :, 1])\n    x2 = np.minimum(a[:, None, 2], b[None, :, 2])\n    y2 = np.minimum(a[:, None, 3], b[None, :, 3])\n    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)\n    area_a = np.maximum(0, a[:, 2] - a[:, 0]) * np.maximum(0, a[:, 3] - a[:, 1])\n    area_b = np.maximum(0, b[:, 2] - b[:, 0]) * np.maximum(0, b[:, 3] - b[:, 1])\n    union = area_a[:, None] + area_b[None, :] - inter\n    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)\n\n\ndef predict(model: YOLO, image_paths: list[Path], conf: float, nms_iou: float, imgsz: int, device: str | None) -> pd.DataFrame:\n    kwargs = {\n        "source": [str(p) for p in image_paths],\n        "imgsz": imgsz,\n        "conf": conf,\n        "iou": nms_iou,\n        "verbose": False,\n        "save": False,\n        "stream": False,\n    }\n    if device:\n        kwargs["device"] = device\n    results = model.predict(**kwargs)\n    rows = []\n    for result_idx, result in enumerate(results):\n        image_key = str(image_paths[result_idx].resolve())\n        if result.boxes is None or len(result.boxes) == 0:\n            continue\n        xyxy = result.boxes.xyxy.cpu().numpy()\n        confs = result.boxes.conf.cpu().numpy()\n        for pred_idx, (box, score) in enumerate(zip(xyxy, confs)):\n            rows.append(\n                {\n                    "image_key": image_key,\n                    "pred_id": f"{Path(image_key).name}:{pred_idx}",\n                    "x1": float(box[0]),\n                    "y1": float(box[1]),\n                    "x2": float(box[2]),\n                    "y2": float(box[3]),\n                    "confidence": float(score),\n                }\n            )\n    columns = ["image_key", "pred_id", "x1", "y1", "x2", "y2", "confidence"]\n    return pd.DataFrame(rows, columns=columns)\n\n\ndef match_gt(gt: pd.DataFrame, predictions: pd.DataFrame, best_predictions: pd.DataFrame, match_iou: float) -> pd.DataFrame:\n    gt = gt.copy()\n    gt["is_found"] = False\n    gt["matched_prediction_confidence"] = np.nan\n    gt["matched_prediction_iou"] = 0.0\n    gt["best_prediction_confidence"] = np.nan\n    gt["best_prediction_iou"] = 0.0\n    pred_by_image = {key: group.sort_values("confidence", ascending=False).reset_index(drop=True) for key, group in predictions.groupby("image_key")} if not predictions.empty else {}\n    best_pred_by_image = {key: group.sort_values("confidence", ascending=False).reset_index(drop=True) for key, group in best_predictions.groupby("image_key")} if not best_predictions.empty else {}\n\n    for image_key, gt_group in gt.groupby("image_key"):\n        pred_group = pred_by_image.get(image_key, pd.DataFrame()).copy()\n        best_pred_group = best_pred_by_image.get(image_key, pd.DataFrame()).copy()\n        gt_indices = list(gt_group.index)\n        gt_boxes = np.array(gt_group["bbox_xyxy"].tolist(), dtype=float)\n        pred_boxes = pred_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float) if not pred_group.empty else np.empty((0, 4))\n        best_pred_boxes = best_pred_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float) if not best_pred_group.empty else np.empty((0, 4))\n        ious = box_iou(pred_boxes, gt_boxes)\n        best_ious = box_iou(best_pred_boxes, gt_boxes)\n\n        if len(best_pred_boxes):\n            for local_gt_idx, global_gt_idx in enumerate(gt_indices):\n                best_pred_idx = int(np.argmax(best_ious[:, local_gt_idx]))\n                gt.loc[global_gt_idx, "best_prediction_iou"] = float(best_ious[best_pred_idx, local_gt_idx])\n                gt.loc[global_gt_idx, "best_prediction_confidence"] = float(best_pred_group.iloc[best_pred_idx]["confidence"])\n\n        candidates = []\n        for pred_idx in range(len(pred_boxes)):\n            for local_gt_idx, global_gt_idx in enumerate(gt_indices):\n                iou_value = float(ious[pred_idx, local_gt_idx])\n                if iou_value >= match_iou:\n                    candidates.append((iou_value, float(pred_group.iloc[pred_idx]["confidence"]), pred_idx, global_gt_idx))\n\n        matched_pred = set()\n        matched_gt = set()\n        for iou_value, confidence, pred_idx, global_gt_idx in sorted(candidates, reverse=True):\n            if pred_idx in matched_pred or global_gt_idx in matched_gt:\n                continue\n            matched_pred.add(pred_idx)\n            matched_gt.add(global_gt_idx)\n            gt.loc[global_gt_idx, "is_found"] = True\n            gt.loc[global_gt_idx, "matched_prediction_iou"] = iou_value\n            gt.loc[global_gt_idx, "matched_prediction_confidence"] = confidence\n    return gt\n\n\ndef add_fn_types(audit: pd.DataFrame, conf_threshold: float, match_iou: float, near_miss_iou: float) -> pd.DataFrame:\n    audit = audit.copy()\n    best_iou = pd.to_numeric(audit["best_prediction_iou"], errors="coerce").fillna(0.0)\n    best_conf = pd.to_numeric(audit["best_prediction_confidence"], errors="coerce").fillna(0.0)\n    audit["fn_type"] = "found"\n    missed = ~audit["is_found"].astype(bool)\n    metric_miss = missed & (best_iou >= match_iou) & (best_conf < conf_threshold)\n    near_miss = missed & (~metric_miss) & (best_iou >= near_miss_iou)\n    hard_miss = missed & (~metric_miss) & (~near_miss)\n    audit.loc[metric_miss, "fn_type"] = "metric_miss"\n    audit.loc[near_miss, "fn_type"] = "near_miss"\n    audit.loc[hard_miss, "fn_type"] = "hard_miss"\n    return audit\n\n\ndef add_features(audit: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n    audit = audit.copy()\n    area = pd.to_numeric(audit["bbox_area_px"], errors="coerce")\n    width = pd.to_numeric(audit["bbox_width_px"], errors="coerce")\n    height = pd.to_numeric(audit["bbox_height_px"], errors="coerce")\n    objects = pd.to_numeric(audit["objects_in_tile"], errors="coerce")\n    median_area = float(area.median())\n    dense_threshold = max(3.0, float(objects.quantile(0.75)))\n\n    q = area.quantile([0.2, 0.4, 0.6, 0.8]).to_list()\n    # Duplicate quantiles can happen on tiny samples; rank fallback keeps buckets usable.\n    if len(set(round(x, 6) for x in q if pd.notna(x))) < 4:\n        audit["size_bucket"] = pd.qcut(area.rank(method="first"), 5, labels=["tiny", "small", "medium", "large", "huge"])\n    else:\n        audit["size_bucket"] = pd.cut(area, bins=[-np.inf, *q, np.inf], labels=["tiny", "small", "medium", "large", "huge"])\n\n    audit["edge_object"] = audit["touches_tile_edge"].astype(bool)\n    audit["dense_cluster"] = objects >= dense_threshold\n    audit["small_object"] = area < median_area\n    audit["large_object"] = area > median_area\n    audit["isolated_object"] = objects <= 1\n    audit["suspicious_fn"] = (~audit["is_found"]) & (area > median_area) & (~audit["edge_object"]) & (~audit["dense_cluster"])\n\n    stats = pd.DataFrame(\n        [\n            {"metric": "bbox_area_median", "value": median_area},\n            {"metric": "bbox_width_median", "value": float(width.median())},\n            {"metric": "bbox_height_median", "value": float(height.median())},\n            {"metric": "dense_cluster_objects_threshold", "value": dense_threshold},\n            {"metric": "size_bucket_q20", "value": float(area.quantile(0.2))},\n            {"metric": "size_bucket_q40", "value": float(area.quantile(0.4))},\n            {"metric": "size_bucket_q60", "value": float(area.quantile(0.6))},\n            {"metric": "size_bucket_q80", "value": float(area.quantile(0.8))},\n        ]\n    )\n    return audit, stats\n\n\ndef crop_with_header(row: pd.Series, out_path: Path, pad_ratio: float = 0.25, min_size: int = 180) -> Image.Image:\n    image = Image.open(row["image_key"]).convert("RGB")\n    w, h = image.size\n    x1, y1, x2, y2 = map(float, row["bbox_xyxy"])\n    bw = x2 - x1\n    bh = y2 - y1\n    pad = max(bw, bh) * pad_ratio\n    cx1 = max(0, int(x1 - pad))\n    cy1 = max(0, int(y1 - pad))\n    cx2 = min(w, int(x2 + pad))\n    cy2 = min(h, int(y2 + pad))\n    crop = image.crop((cx1, cy1, cx2, cy2))\n    scale = max(1.0, min_size / max(crop.width, crop.height))\n    if scale > 1.0:\n        crop = crop.resize((int(crop.width * scale), int(crop.height * scale)), Image.Resampling.LANCZOS)\n    header_h = 46\n    canvas = Image.new("RGB", (crop.width, crop.height + header_h), "white")\n    canvas.paste(crop, (0, header_h))\n    draw = ImageDraw.Draw(canvas)\n    font = ImageFont.load_default()\n    label = f"{row[\'image_id\']} | {row[\'region\']} | {row[\'source_class_name\']} | area={row[\'bbox_area_px\']:.0f}"\n    draw.rectangle([0, 0, canvas.width, header_h], fill="black")\n    draw.text((4, 4), label[:120], fill="white", font=font)\n    conf_text = f"{row[\'best_prediction_confidence\']:.3f}" if pd.notna(row["best_prediction_confidence"]) else "NA"\n    draw.text((4, 22), f"{row[\'fn_type\']} | best IoU={row[\'best_prediction_iou\']:.3f} conf={conf_text}", fill="white", font=font)\n    # Draw GT bbox position inside crop after scaling.\n    sx = crop.width / max(1, cx2 - cx1)\n    sy = crop.height / max(1, cy2 - cy1)\n    box = [(x1 - cx1) * sx, header_h + (y1 - cy1) * sy, (x2 - cx1) * sx, header_h + (y2 - cy1) * sy]\n    color = CLASS_COLORS.get(str(row["source_class_name"]), "#00ff66")\n    draw.rectangle(box, outline=color, width=3)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n    return canvas\n\n\ndef write_contact_sheets(crop_paths: list[Path], out_dir: Path, prefix: str = "fn_page", page_size: int = PAGE_SIZE, columns: int = 5) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for page_idx in range(math.ceil(len(crop_paths) / page_size)):\n        paths = crop_paths[page_idx * page_size : (page_idx + 1) * page_size]\n        images = [Image.open(path).convert("RGB") for path in paths]\n        if not images:\n            continue\n        cell_w = max(img.width for img in images)\n        cell_h = max(img.height for img in images)\n        rows = math.ceil(len(images) / columns)\n        pad = 10\n        sheet = Image.new("RGB", (columns * cell_w + (columns + 1) * pad, rows * cell_h + (rows + 1) * pad), "white")\n        for idx, img in enumerate(images):\n            x = pad + (idx % columns) * (cell_w + pad)\n            y = pad + (idx // columns) * (cell_h + pad)\n            sheet.paste(img, (x, y))\n        sheet.save(out_dir / f"{prefix}_{page_idx + 1:02d}.jpg", quality=92)\n\n\ndef write_plots(audit: pd.DataFrame, fn: pd.DataFrame, out_dir: Path) -> None:\n    if plt is None:\n        return\n    out_dir.mkdir(parents=True, exist_ok=True)\n    found = audit[audit["is_found"]].copy()\n    missed = audit[~audit["is_found"]].copy()\n    for metric in ["bbox_area_px", "bbox_width_px", "bbox_height_px"]:\n        fig, ax = plt.subplots(figsize=(8, 5))\n        for label, df in [("FOUND", found), ("MISSED", missed)]:\n            values = pd.to_numeric(df[metric], errors="coerce").dropna()\n            if not values.empty:\n                ax.hist(values, bins=24, alpha=0.55, label=label)\n        ax.set_title(f"{metric}: FOUND vs MISSED")\n        ax.set_xlabel(metric)\n        ax.set_ylabel("GT objects")\n        ax.grid(True, alpha=0.25)\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out_dir / f"found_vs_missed_{metric}.png", dpi=180)\n        plt.close(fig)\n\n    for col in ["source_class_name", "region", "size_bucket"]:\n        counts = fn[col].astype(str).value_counts().sort_values(ascending=True)\n        fig, ax = plt.subplots(figsize=(9, max(4, len(counts) * 0.35)))\n        counts.plot(kind="barh", ax=ax)\n        ax.set_title(f"False negatives by {col}")\n        ax.set_xlabel("count")\n        ax.grid(True, axis="x", alpha=0.25)\n        fig.tight_layout()\n        fig.savefig(out_dir / f"fn_by_{col}.png", dpi=180)\n        plt.close(fig)\n\n\ndef describe_found_vs_missed(audit: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for label, df in [("FOUND", audit[audit["is_found"]]), ("MISSED", audit[~audit["is_found"]])]:\n        for metric in ["bbox_area_px", "bbox_width_px", "bbox_height_px"]:\n            values = pd.to_numeric(df[metric], errors="coerce").dropna()\n            if values.empty:\n                continue\n            rows.append(\n                {\n                    "group": label,\n                    "metric": metric,\n                    "count": int(len(values)),\n                    "mean": float(values.mean()),\n                    "median": float(values.median()),\n                    "p10": float(values.quantile(0.10)),\n                    "p25": float(values.quantile(0.25)),\n                    "p75": float(values.quantile(0.75)),\n                    "p90": float(values.quantile(0.90)),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef reason_candidates(fn: pd.DataFrame) -> pd.DataFrame:\n    reason_cols = ["small_object", "edge_object", "dense_cluster", "large_object", "isolated_object"]\n    rows = [{"reason_candidate": col, "count": int(fn[col].sum())} for col in reason_cols]\n    return pd.DataFrame(rows).sort_values("count", ascending=False)\n\n\ndef fn_type_summary(fn: pd.DataFrame) -> pd.DataFrame:\n    if fn.empty:\n        return pd.DataFrame(columns=["fn_type", "count"])\n    return fn["fn_type"].astype(str).value_counts().rename_axis("fn_type").reset_index(name="count")\n\n\ndef priority_images(fn: pd.DataFrame) -> pd.DataFrame:\n    by_image = (\n        fn.groupby(["image_id", "image_key", "region"], dropna=False)\n        .agg(\n            fn_count=("gt_id", "count"),\n            suspicious_fn_count=("suspicious_fn", "sum"),\n            median_fn_area=("bbox_area_px", "median"),\n            max_fn_area=("bbox_area_px", "max"),\n            source_classes=("source_class_name", lambda s: "; ".join(sorted(set(map(str, s))))),\n        )\n        .reset_index()\n    )\n    return by_image.sort_values(["suspicious_fn_count", "fn_count", "max_fn_area"], ascending=False).head(20)\n\n\ndef markdown_table(df: pd.DataFrame, floatfmt: str = ".3f") -> str:\n    if df.empty:\n        return "_No rows._"\n    formatted = df.copy()\n    for col in formatted.columns:\n        if pd.api.types.is_float_dtype(formatted[col]):\n            formatted[col] = formatted[col].map(lambda x: format(x, floatfmt))\n    formatted = formatted.fillna("")\n    cols = list(formatted.columns)\n    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]\n    for _, row in formatted.iterrows():\n        lines.append("| " + " | ".join(str(row[col]) for col in cols) + " |")\n    return "\\n".join(lines)\n\n\ndef write_summary(\n    out_path: Path,\n    audit: pd.DataFrame,\n    fn: pd.DataFrame,\n    size_stats: pd.DataFrame,\n    reasons: pd.DataFrame,\n    fn_types: pd.DataFrame,\n    priority: pd.DataFrame,\n    conf_threshold: float,\n    match_iou: float,\n    near_miss_iou: float,\n) -> None:\n    found = audit[audit["is_found"]]\n    recall = len(found) / len(audit) if len(audit) else 0.0\n    top_reason = reasons.iloc[0]["reason_candidate"] if not reasons.empty else "NA"\n    fn_by_region = fn["region"].astype(str).value_counts().reset_index()\n    fn_by_region.columns = ["region", "fn_count"]\n    fn_by_class = fn["source_class_name"].astype(str).value_counts().reset_index()\n    fn_by_class.columns = ["source_class_name", "fn_count"]\n    fn_by_size = fn["size_bucket"].astype(str).value_counts().reset_index()\n    fn_by_size.columns = ["size_bucket", "fn_count"]\n    found_median = size_stats[(size_stats["group"].eq("FOUND")) & (size_stats["metric"].eq("bbox_area_px"))]["median"]\n    missed_median = size_stats[(size_stats["group"].eq("MISSED")) & (size_stats["metric"].eq("bbox_area_px"))]["median"]\n\n    if not found_median.empty and not missed_median.empty:\n        size_answer = "маленькими объектами" if float(missed_median.iloc[0]) < float(found_median.iloc[0]) else "не только маленькими объектами"\n    else:\n        size_answer = "недостаточно данных для вывода"\n    dense_share = float(fn["dense_cluster"].mean()) if len(fn) else 0.0\n    dense_answer = "да" if dense_share >= 0.5 else "нет"\n\n    lines = [\n        "# v3h False Negative Audit",\n        "",\n        f"GT objects: `{len(audit)}`",\n        f"Found: `{len(found)}`",\n        f"Missed: `{len(fn)}`",\n        f"Object-level recall at audit threshold: `{recall:.3f}`",\n        f"FOUND/MISSED rule: `confidence >= {conf_threshold}` and `IoU >= {match_iou}`.",\n        f"FN typing rule: `metric_miss` = IoU >= {match_iou} but confidence < {conf_threshold}; `near_miss` = IoU >= {near_miss_iou} but not FOUND; `hard_miss` = IoU < {near_miss_iou}.",\n        "",\n        "## FOUND vs MISSED Size Statistics",\n        "",\n        markdown_table(size_stats),\n        "",\n        "## False Negatives By Source Class",\n        "",\n        markdown_table(fn_by_class),\n        "",\n        "## False Negatives By Region",\n        "",\n        markdown_table(fn_by_region.head(20)),\n        "",\n        "## False Negatives By Size Bucket",\n        "",\n        markdown_table(fn_by_size),\n        "",\n        "## Reason Candidates",\n        "",\n        markdown_table(reasons),\n        "",\n        "## False Negative Types",\n        "",\n        markdown_table(fn_types),\n        "",\n        "## 20 Images For Manual Review",\n        "",\n        markdown_table(priority[["image_id", "region", "fn_count", "suspicious_fn_count", "median_fn_area", "max_fn_area", "source_classes"]]),\n        "",\n        "## Answers",\n        "",\n        f"- Самая массовая группа FN: `{top_reason}`.",\n        f"- Recall ограничивается {size_answer}; см. медианы FOUND/MISSED выше.",\n        f"- Recall ограничивается плотными кластерами: `{dense_answer}` (`dense_cluster` share among FN = `{dense_share:.3f}`).",\n        f"- Регионы с наибольшим числом FN: {\', \'.join(fn_by_region.head(5)[\'region\'].astype(str).tolist())}.",\n        "- Подозрительные FN сохранены в `outputs/suspicious_fn/`: это крупные, не edge, не dense-cluster объекты, которые модель должна была увидеть.",\n        "",\n    ]\n    out_path.write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef main() -> None:\n    args = parse_args()\n    args.out_dir.mkdir(parents=True, exist_ok=True)\n    outputs_dir = args.out_dir / "outputs"\n    crops_dir = outputs_dir / "fn_crops"\n    suspicious_dir = outputs_dir / "suspicious_fn"\n    sheets_dir = outputs_dir / "fn_contact_sheets"\n    plots_dir = args.out_dir / "plots"\n    for path in [crops_dir, suspicious_dir, sheets_dir, plots_dir]:\n        path.mkdir(parents=True, exist_ok=True)\n\n    _, gt, image_paths = load_val_metadata(args.metadata)\n    model = YOLO(str(args.weights))\n    all_predictions = predict(model, image_paths, args.analysis_conf, args.nms_iou, args.imgsz, args.device)\n    predictions = all_predictions[pd.to_numeric(all_predictions["confidence"], errors="coerce") >= args.conf].copy() if not all_predictions.empty else all_predictions.copy()\n    all_predictions.to_csv(args.out_dir / "predictions_all_conf.csv", index=False)\n    predictions.to_csv(args.out_dir / "predictions.csv", index=False)\n    audit = match_gt(gt, predictions, all_predictions, args.match_iou)\n    audit = add_fn_types(audit, args.conf, args.match_iou, args.near_miss_iou)\n    audit, feature_stats = add_features(audit)\n\n    export_cols = [\n        "gt_id",\n        "image_id",\n        "image_key",\n        "is_found",\n        "matched_prediction_confidence",\n        "matched_prediction_iou",\n        "best_prediction_confidence",\n        "best_prediction_iou",\n        "fn_type",\n        "bbox_area_px",\n        "bbox_width_px",\n        "bbox_height_px",\n        "source_class_name",\n        "region",\n        "source_id",\n        "touches_tile_edge",\n        "objects_in_tile",\n        "size_bucket",\n        "edge_object",\n        "dense_cluster",\n        "small_object",\n        "large_object",\n        "isolated_object",\n        "suspicious_fn",\n    ]\n    audit[export_cols].to_csv(args.out_dir / "gt_object_audit.csv", index=False)\n    fn = audit[~audit["is_found"]].copy()\n    found = audit[audit["is_found"]].copy()\n    fn[export_cols].to_csv(args.out_dir / "false_negative_audit.csv", index=False)\n    found[export_cols].to_csv(args.out_dir / "found_object_audit.csv", index=False)\n\n    size_stats = describe_found_vs_missed(audit)\n    reasons = reason_candidates(fn)\n    fn_types = fn_type_summary(fn)\n    priority = priority_images(fn)\n    size_stats.to_csv(args.out_dir / "found_vs_missed_size_stats.csv", index=False)\n    feature_stats.to_csv(args.out_dir / "feature_thresholds.csv", index=False)\n    reasons.to_csv(args.out_dir / "reason_candidates.csv", index=False)\n    fn_types.to_csv(args.out_dir / "false_negative_types.csv", index=False)\n    priority.to_csv(args.out_dir / "manual_review_priority_images.csv", index=False)\n\n    fn_crop_paths: list[Path] = []\n    for _, row in fn.sort_values(["suspicious_fn", "bbox_area_px"], ascending=False).iterrows():\n        crop_name = f"fn_gt_{int(row[\'gt_id\']):04d}_{Path(str(row[\'image_id\'])).stem}.jpg"\n        crop_path = crops_dir / crop_name\n        crop_with_header(row, crop_path)\n        fn_crop_paths.append(crop_path)\n        if bool(row["suspicious_fn"]):\n            crop_with_header(row, suspicious_dir / crop_name)\n\n    write_contact_sheets(fn_crop_paths, sheets_dir)\n    write_plots(audit, fn, plots_dir)\n    write_summary(args.out_dir / "summary.md", audit, fn, size_stats, reasons, fn_types, priority, args.conf, args.match_iou, args.near_miss_iou)\n\n    print("Saved audit to:", args.out_dir)\n    print("GT:", len(audit), "found:", len(found), "missed:", len(fn))\n    print("FN crops:", crops_dir)\n    print("Suspicious FN:", suspicious_dir)\n    print("Summary:", args.out_dir / "summary.md")\n\n\nif __name__ == "__main__":\n    main()\n'

audit_script_path = SCRIPT_DIR / "audit_v3h_false_negatives.py"
audit_script_path.write_text(AUDIT_SCRIPT, encoding="utf-8")
print("Audit script:", audit_script_path)


## 5. Run False Negative Audit


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(SCRIPT_DIR / "audit_v3h_false_negatives.py"),
    "--metadata",
    str(METADATA_PATH),
    "--weights",
    str(weights_path),
    "--out-dir",
    str(OUTPUT_ROOT),
    "--imgsz",
    str(IMGSZ),
    "--conf",
    str(CONF),
    "--analysis-conf",
    str(ANALYSIS_CONF),
    "--nms-iou",
    str(NMS_IOU),
    "--match-iou",
    str(MATCH_IOU),
    "--near-miss-iou",
    str(NEAR_MISS_IOU),
    "--device",
    str(DEVICE),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 6. Inspect Tables


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

audit = pd.read_csv(OUTPUT_ROOT / "gt_object_audit.csv")
fn = pd.read_csv(OUTPUT_ROOT / "false_negative_audit.csv")
size_stats = pd.read_csv(OUTPUT_ROOT / "found_vs_missed_size_stats.csv")
reasons = pd.read_csv(OUTPUT_ROOT / "reason_candidates.csv")
fn_types = pd.read_csv(OUTPUT_ROOT / "false_negative_types.csv")
priority = pd.read_csv(OUTPUT_ROOT / "manual_review_priority_images.csv")

display(Markdown("### GT Object Audit"))
display(audit.head())
display(Markdown("### FOUND vs MISSED size stats"))
display(size_stats)
display(Markdown("### Reason candidates"))
display(reasons)
display(Markdown("### False negative types"))
display(fn_types)
display(Markdown("### False negative audit with best low-confidence prediction"))
display(fn[[
    "image_id",
    "region",
    "source_class_name",
    "bbox_area_px",
    "best_prediction_iou",
    "best_prediction_confidence",
    "fn_type",
]].head(25))
display(Markdown("### 20 images to review first"))
display(priority)
display(Markdown(f"False negatives: `{len(fn)}`"))


## 7. Show Plots


In [ ]:
from IPython.display import Image, display

for path in sorted((OUTPUT_ROOT / "plots").glob("*.png")):
    print(path)
    display(Image(filename=str(path)))


## 8. Show Contact Sheets


In [ ]:
from IPython.display import Image, display

sheets = sorted((OUTPUT_ROOT / "outputs" / "fn_contact_sheets").glob("fn_page_*.jpg"))
for sheet in sheets[:6]:
    print(sheet)
    display(Image(filename=str(sheet)))
print("Total contact sheets:", len(sheets))


## 9. Show Suspicious FN Examples


In [ ]:
from IPython.display import Image, display

suspicious = sorted((OUTPUT_ROOT / "outputs" / "suspicious_fn").glob("*.jpg"))
for crop in suspicious[:20]:
    print(crop)
    display(Image(filename=str(crop)))
print("Suspicious FN crops:", len(suspicious))


## 10. Summary


In [ ]:
from IPython.display import Markdown, display

summary_text = (OUTPUT_ROOT / "summary.md").read_text(encoding="utf-8")
display(Markdown(summary_text))


## 11. Archive Outputs


In [ ]:
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = WORK_ROOT / f"v3h_false_negative_audit_{timestamp}.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in OUTPUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(WORK_ROOT))

print("Archive:", archive_path)
print("Archive size MB:", round(archive_path.stat().st_size / (1024 * 1024), 2))
